# 02 — Fine-tuning com LoRA

**Ateliê Generativo — Arquitetura Modernista de Brasília**

Este notebook faz a Etapa 2 do projeto:
1. Prepara o ambiente e clona o script oficial do `diffusers`
2. Converte seu dataset (imagens + `legendas.txt`) para o formato que o script de treino espera
3. Treina **duas configurações distintas** de LoRA (rank e learning rate diferentes) e publica no Hugging Face Hub

⚠️ Antes de rodar: ative a GPU em **Runtime → Change runtime type → T4 GPU**.

⚠️ **Sobre desconexões do Colab:** cada treino leva 40-90 minutos. O Colab desconecta por inatividade (~90 min sem interação com a aba). Mantenha a aba aberta e visível, e não deixe o computador dormir. Se desconectar no meio, não é o fim do mundo — o Passo 5 explica como retomar de um checkpoint salvo, em vez de perder o progresso e começar do zero.

## Passo 1 — Trazer seu dataset do GitHub

In [ ]:
!git clone https://github.com/lramosc1512/atelie-generativo-LeonidIA.git
%cd atelie-generativo-LeonidIA
!ls dados/imagens | wc -l
!head -3 dados/legendas.txt

## Passo 2 — Instalar dependências e clonar o script oficial de treino

In [ ]:
!pip -q install diffusers transformers accelerate peft datasets
%cd /content
!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/text_to_image
!pip -q install -r requirements.txt

## Passo 3 — Autenticar no Hugging Face

Necessário para baixar o Stable Diffusion base e para publicar os pesos do LoRA no seu Hub depois.

In [ ]:
from huggingface_hub import login
login()

## Passo 4 — Preparar o dataset no formato esperado pelo script

O `train_text_to_image_lora.py` espera uma pasta com as imagens e um arquivo `metadata.jsonl` dentro dela, no formato `{"file_name": "img_001.jpg", "text": "legenda"}` por linha (padrão `imagefolder` da biblioteca `datasets`). Essa célula converte seu `legendas.txt` para esse formato automaticamente.

In [ ]:
import json
import shutil
from pathlib import Path

ORIGEM_IMAGENS = Path("/content/atelie-generativo-LeonidIA/dados/imagens")
LEGENDAS_TXT = Path("/content/atelie-generativo-LeonidIA/dados/legendas.txt")
PASTA_DATASET = Path("/content/dataset_estilo_brasilia")

PASTA_DATASET.mkdir(parents=True, exist_ok=True)

linhas_metadata = []
with open(LEGENDAS_TXT, "r", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:
            continue
        arquivo, legenda = linha.split(",", 1)
        origem = ORIGEM_IMAGENS / arquivo
        destino = PASTA_DATASET / arquivo
        shutil.copy(origem, destino)
        linhas_metadata.append({"file_name": arquivo, "text": legenda.strip()})

with open(PASTA_DATASET / "metadata.jsonl", "w", encoding="utf-8") as f:
    for item in linhas_metadata:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"{len(linhas_metadata)} imagens copiadas e metadata.jsonl gerado em {PASTA_DATASET}")

## Passo 5 — Configuração 1: rank baixo, learning rate mais alto

**Hipótese a testar:** rank menor (4) captura menos detalhe do estilo, mas treina mais rápido e tem menor risco de overfitting num dataset pequeno como o nosso (50 imagens).

Troque `lramosc1512` pelo seu nome de usuário do Hugging Face antes de rodar.

**Se o Colab desconectar no meio do treino:** rode esta mesma célula de novo, adicionando `--resume_from_checkpoint="latest"` — ela retoma do último checkpoint salvo em vez de recomeçar.

In [ ]:
%cd /content/diffusers/examples/text_to_image

!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset_estilo_brasilia" \
  --resolution=512 --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-4 --lr_scheduler="cosine" \
  --rank=4 \
  --mixed_precision="fp16" \
  --checkpointing_steps=250 \
  --validation_prompt="estilo_brasilia, fachada curva de concreto branco refletida em espelho d'agua ao entardecer" \
  --output_dir="/content/lora_config1_rank4" \
  --push_to_hub --hub_model_id="lramosc1512/lora-estilo-brasilia-config1"

## Passo 6 — Configuração 2: rank mais alto, learning rate mais baixo

**Hipótese a testar:** rank maior (8) tem mais capacidade para capturar detalhes finos do estilo (as curvas específicas de cada prédio, texturas de concreto), ao custo de mais risco de overfitting — por isso, compensamos com learning rate mais conservador e mais passos.

Rode isso **depois** que a Configuração 1 terminar (ou em uma sessão separada) — rodar as duas ao mesmo tempo na mesma GPU T4 não é possível.

In [ ]:
%cd /content/diffusers/examples/text_to_image

!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset_estilo_brasilia" \
  --resolution=512 --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=5e-5 --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=250 \
  --validation_prompt="estilo_brasilia, fachada curva de concreto branco refletida em espelho d'agua ao entardecer" \
  --output_dir="/content/lora_config2_rank8" \
  --push_to_hub --hub_model_id="lramosc1512/lora-estilo-brasilia-config2"

## Passo 7 — Conferir as imagens de validação de cada checkpoint

Durante o treino, o script gera uma imagem de validação a cada `checkpointing_steps` (a cada 250 passos, no nosso caso) usando o `validation_prompt`. Essas imagens ficam salvas dentro da pasta de output e também aparecem nos logs do treino acima — role para cima e observe como o estilo evolui (ou começa a decorar demais o dataset) a cada checkpoint. Isso alimenta diretamente a Etapa 3 (verificação de overfitting).

## Passo 8 — Salvar os checkpoints no Google Drive (backup)

O armazenamento do Colab é temporário — some quando a sessão encerra. Como os pesos já foram enviados ao Hub (`--push_to_hub`), isso já é seu backup principal. Mas para não depender só disso, vale copiar também para o Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copytree("/content/lora_config1_rank4", "/content/drive/MyDrive/atelie_generativo/lora_config1_rank4", dirs_exist_ok=True)
shutil.copytree("/content/lora_config2_rank8", "/content/drive/MyDrive/atelie_generativo/lora_config2_rank8", dirs_exist_ok=True)
print("Checkpoints copiados para o Google Drive.")

## Próximos passos

- Confirme no seu perfil do Hugging Face que os dois modelos (`lora-estilo-brasilia-config1` e `-config2`) aparecem publicados
- Preencha o **model card** de cada um no Hub (descrição do estilo, dataset, hiperparâmetros, limitações conhecidas) — exigido pelo Critério 5 do barema
- Guarde as imagens de validação de cada checkpoint — vão para a grade comparativa da Etapa 3
- Isso não sobe para o Git (pesos ficam no Hub, não no repositório) — só o notebook em si:

```bash
git add notebooks/02_treino_lora.ipynb
git commit -m "Adiciona notebook de treino LoRA (2 configuracoes)"
git push
```